# Unsupervised Learning: k-Means, Hierarchical Clustering and PCA – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. Unsupervised learning: no target variable
2. k-means clustering
3. Choosing k: elbow method and silhouette analysis
4. Hierarchical clustering and the dendrogram
5. Principal component analysis (PCA)

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. Unsupervised learning: no target variable

| aspect | supervised | unsupervised |
|:---|:---|:---|
| data | features $x$ **and** known answer $y$ | only features $x$ |
| goal | predict $y$ for new cases | find groups or a compact description |
| evaluation | test error (RMSE, accuracy, AUC) | no target used for fitting: internal criteria, stability, interpretation (external labels sometimes available) |
| typical use | rent prediction, churn yes/no | customer segments, anomaly detection, visualisation |

- **Clustering:** group similar observations – k-means, hierarchical clustering.
- **Dimension reduction:** summarise many features by a few new ones – PCA.

## 2. k-means clustering

1. Choose the number of clusters $k$.
2. Place $k$ centres at random.
3. Assign every point to its nearest centre.
4. Move each centre to the mean of its points – repeat 3–4 until (almost) nothing changes.

In words: k-means makes the distances of the points to their own cluster centre as small as possible, i.e. it minimises
the **within-cluster sum of squares**

$$\text{WCSS} = \sum_{c=1}^{k}\sum_{x_i \in C_c}\lVert x_i - \mu_c\rVert^2 .$$

In Python: `KMeans(n_clusters=3, n_init=10)`. The result depends on the random start (local optimum) → several starts
(`n_init`), the best solution (smallest WCSS) is kept.

In [ ]:
# Simulate 150 points in three groups
X, _ = make_blobs(
    n_samples=150,
    centers=[[-2, -1.5], [2.5, -1], [0, 2.5]],
    cluster_std=0.8,
    random_state=4,
)


# Function: assigns every point to its nearest centre (cluster 0, 1 or 2)
def nearest_centre(X, centres):
    """Number of the nearest centre for every point."""
    distances = []
    for centre in centres:
        distances.append(np.sqrt(np.sum((X - centre) ** 2, axis=1)))
    return np.argmin(distances, axis=0)


# k-means with a deliberately bad start
centres = np.array([[-1.5, -2.5], [-0.5, -0.5], [2.5, 2.5]])
for iteration in range(21):
    # Step 3: assign every point to its nearest centre
    labels = nearest_centre(X, centres)

    # Figure showing the clusters after 0, 1 and 20 iterations
    if iteration in [0, 1, 20]:
        wcss = np.sum((X - centres[labels]) ** 2)
        plt.scatter(X[:, 0], X[:, 1], c=labels)
        plt.scatter(centres[:, 0], centres[:, 1], s=250, marker='X', c='red')
        plt.title(
            'After '
            + str(iteration)
            + ' iterations: WCSS = '
            + str(round(wcss))
        )
        plt.show()

    # Step 4: move each centre to the mean of its points
    new_centres = []
    for j in range(3):
        new_centres.append(X[labels == j].mean(axis=0))
    centres = np.array(new_centres)

## 3. Choosing k: elbow method and silhouette analysis

**Reading the elbow plot**
- y-axis: how spread out the points are within their clusters (WCSS; smaller = tighter clusters),
- more clusters always reduce it – $k = n$ would give 0,
- choose $k$ where the curve bends (the "elbow").

**Caution:** scale the features first, use several random starts, and check that the clusters make sense for the
business question.

The **silhouette score** (between −1 and 1) measures how well a point fits its own cluster compared with the nearest other cluster; higher is better.

In [ ]:
# WCSS (k = 1..8) and silhouette score (k = 2..8)
ks = range(1, 9)
wcss = []
sil = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(X)
    wcss.append(km.inertia_)
    if k > 1:
        sil.append(silhouette_score(X, labels))

# Figure showing the elbow plot
plt.plot(ks, wcss, marker='o')
plt.scatter(3, wcss[2], s=200, color='orange', label='elbow: k = 3')
plt.xlabel('number of clusters k')
plt.ylabel('within-cluster sum of squares')
plt.title('Elbow plot')
plt.legend()
plt.show()

# Figure showing the mean silhouette score
plt.plot(range(2, 9), sil, marker='o')
plt.xlabel('number of clusters k')
plt.ylabel('mean silhouette score')
plt.title('Silhouette score')
plt.show()

**Silhouette analysis.** The silhouette plot shows the silhouette value of **every** point, sorted within each cluster.
Wide bars above the mean line indicate well-separated clusters; thin clusters, many values close to 0 or negative values
indicate a poor choice of $k$ or badly placed points. Below: $k = 3$ (the true structure) vs. $k = 5$.

In [ ]:
# Silhouette plots for k = 3 and k = 5
for k in [3, 5]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    values = silhouette_samples(X, labels)

    # Points sorted by cluster and by silhouette value
    table = pd.DataFrame({'cluster': labels, 'silhouette': values})
    table = table.sort_values(['cluster', 'silhouette'])

    # Figure: one horizontal bar per point, coloured by cluster
    colors = plt.colormaps['tab10'](table['cluster'])
    plt.barh(range(len(table)), table['silhouette'], color=colors)
    plt.axvline(values.mean(), color='black', linestyle='--', label='mean')
    plt.title(
        'k = ' + str(k) + ': mean silhouette = ' + str(round(values.mean(), 2))
    )
    plt.xlabel('silhouette value')
    plt.yticks([])
    plt.legend()
    plt.show()

## 4. Hierarchical clustering and the dendrogram

**Agglomerative clustering:**
1. Start: every observation (e.g. every customer) is its own cluster.
2. Merge the two closest clusters; repeat until one cluster is left.
3. Cut the **dendrogram** at a height → clusters.

The height of a merge in the dendrogram is the distance between the merged clusters. Long vertical lines mean that
clearly separated groups are merged only late – a good place to cut.

In [ ]:
# Twelve customers in three groups (2 features)
rng = np.random.default_rng(5)
customers = np.vstack(
    [
        rng.normal([1, 1], 0.4, (4, 2)),
        rng.normal([4, 4], 0.3, (4, 2)),
        rng.normal([6, 1.5], 0.4, (4, 2)),
    ]
)
names = list('ABCDEFGHIJKL')

# Figure showing the customers with their names
plt.scatter(customers[:, 0], customers[:, 1], color='black')
for i in range(12):
    plt.text(customers[i, 0] + 0.1, customers[i, 1] + 0.1, names[i])
plt.title('12 customers (2 features)')
plt.show()

# Hierarchical clustering with Ward linkage
Z = linkage(customers, method='ward')

# Figure showing the dendrogram and the cut at height 3
dendrogram(Z, labels=names, color_threshold=3)
plt.axhline(3, color='orange', linestyle='--', label='cut here -> 3 clusters')
plt.ylabel('merge distance')
plt.title('Dendrogram (Ward linkage)')
plt.legend()
plt.show()

# Cluster of each customer after the cut
clusters = fcluster(Z, t=3, criterion='distance')
print(pd.Series(clusters, index=names))

## 5. Principal component analysis (PCA)

**Idea:** PCA rotates the coordinate system so that the first new axis (**PC1**) captures as much of the spread
(variance) as possible; PC2 captures as much as possible of the remaining spread, orthogonal to PC1; and so on.

- The **loadings** (`pca.components_`) show how each principal component is composed of the original features.
- The **explained variance ratio** shows which share of the total variance each component captures.
- Features measured on different scales must be **standardised** first – otherwise PCA is dominated by the features with
  the largest variance.

Example: two strongly correlated features – PC1 explains about 86 % of the total variance, so the two features can be summarised well by one.

In [ ]:
# Two correlated features
rng = np.random.default_rng(0)
f1 = rng.normal(0, 1.8, 200)
f2 = 0.65 * f1 + rng.normal(0, 1.0, 200)
F = np.column_stack([f1, f2])

# Fit the PCA
pca = PCA()
pca.fit(F)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
print('Loadings PC1:', pca.components_[0].round(2))
print('Loadings PC2:', pca.components_[1].round(2))

# Figure showing the data and the two new axes (arrows from the mean)
plt.scatter(F[:, 0], F[:, 1])
for i in range(2):
    arrow = pca.components_[i] * 2 * np.sqrt(pca.explained_variance_[i])
    plt.arrow(
        pca.mean_[0], pca.mean_[1], arrow[0], arrow[1], width=0.08, color='red'
    )
    plt.text(
        pca.mean_[0] + arrow[0],
        pca.mean_[1] + arrow[1],
        'PC' + str(i + 1),
        fontsize=14,
    )
plt.axis('equal')
plt.xlabel('feature 1')
plt.ylabel('feature 2')
plt.title('New axes along the largest spread')
plt.show()

# Figure showing the explained variance
plt.bar(['PC1', 'PC2'], pca.explained_variance_ratio_ * 100)
plt.ylabel('share of total variance [%]')
plt.title('Explained variance')
plt.show()

**Key messages**
- Unsupervised learning has no target: results are judged by internal criteria, stability and interpretation.
- k-means minimises the WCSS; scale the features, use several starts and choose $k$ with the elbow plot, the silhouette
  score and domain knowledge.
- Hierarchical clustering shows the whole merge history in a dendrogram; cutting it gives the clusters.
- PCA summarises many correlated features by a few principal components.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')